# Global seismicity map — one year of M5+

Map a full year of global **M5+** earthquakes from USGS, with marker area scaled by magnitude and colour by hypocentre depth — the classic view that traces the plate boundaries.

## Setup

The imports up front: `Path` for the scratch output directory, `matplotlib` for the map, and the unified `EarthLens` entry point. FDSN is a *vector* backend, so `download()` returns a `GeoDataFrame` of events.

In [ ]:
# Off the system drive; override with EARTHLENS_CACHE if your cache lives elsewhere.
import os
from pathlib import Path

import digitalearth as de
import matplotlib.pyplot as plt
from digitalearth.static.animation import save_animation
from IPython.display import Image, display

from earthlens.core import EarthLens

OUT_DIR = Path(os.environ.get('EARTHLENS_CACHE', 'D:/earthlens-cache')) / 'fdsn'
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Query one year of global M5+ events

First build the request — source, catalogue (`variables`), the 2023 date window, a whole-globe bounding box, and a magnitude floor. Keeping construction on its own line makes the parameters easy to read and tweak.

In [ ]:
quake = EarthLens(
    variables=['USGS'],
    data_source='fdsn',
    start='2023-01-01',
    end='2023-12-31',
    aoi=[-180, -90, 180, 90],
    min_magnitude=5.0,
    path=OUT_DIR,
)

The `download()` call performs the live FDSN query and returns the events into memory. It is wrapped in a `try`/`except` so the notebook degrades gracefully (rather than erroring) when run offline.

In [ ]:
events = None
try:
    events = quake.download()
    print(f'{len(events)} M5+ events in 2023')
except Exception as exc:
    print(f'live FDSN query skipped (offline?): {type(exc).__name__}: {exc}')

## Magnitude-scaled, depth-coloured scatter

Plot each epicentre with marker area growing as `2 ** magnitude` and colour set by hypocentre depth, so the deep subduction-zone events stand out from the shallow ridge seismicity. The plot is guarded on a successful download.

In [ ]:
if events is not None and len(events):
    fig, ax = plt.subplots(figsize=(12, 6))
    sc = ax.scatter(
        events['longitude'],
        events['latitude'],
        s=2 ** events['magnitude'],
        c=events['depth_km'],
        cmap='viridis_r',
        alpha=0.6,
        edgecolor='none',
    )
    fig.colorbar(sc, ax=ax, label='depth (km)')
    ax.set_title('Global M5+ seismicity, 2023 (marker area ~ magnitude)')
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)

## The same events on a globe — Digital-Earth's 3-D tier

The scatter above is drawn in raw lon/lat, so the plate boundaries it traces are stretched by the flat
projection: Alaska and the Andes end up the same width, and the ring around the Pacific is cut in half at
the antimeridian.

`digitalearth.TexturedGlobe` drapes an XYZ imagery basemap over a real sphere and scatters the epicentres
onto it, so the ring closes and the arcs keep their true shape. `from_provider` fetches the whole-world
texture once; `points` takes the `GeoDataFrame` the FDSN backend returned and culls whatever falls on the
far side, so nothing shows through the planet.

Which side faces the viewer matters more than it looks. The seismicity traces the *rim* of the Pacific,
so centring the ocean itself pushes every arc onto the limb where it foreshortens into a thin edge; the
view below is turned to the western Pacific, where half the year's events are.

This is the part of Digital-Earth that has no pyramids equivalent — pyramids plots a raster or a
`FeatureCollection` onto a 2-D axes, and stops there.

In [ ]:
if events is not None and len(events):
    magnitude = events['magnitude'].to_numpy()
    depth = events['depth_km'].to_numpy()

    globe = de.TexturedGlobe.from_provider(
        'Esri.WorldImagery', texture_n_lon=1440, texture_n_lat=720, n_lon=360, n_lat=180
    )
    # The visible meridian is about 6 - spin, so 231 centres ~135E: Japan, the
    # Philippines, Indonesia and Tonga. That is where the events actually are --
    # 877 of the 1778 sit between +120 and +180. Centring the open Pacific instead
    # puts the whole ring of fire on the limb, edge-on and unreadable.
    globe.draw(spin=231.0)
    quakes = globe.points(
        events,
        s=2**magnitude / 2,  # marker area ~ magnitude, as in the flat map above
        c=depth,
        cmap='viridis_r',
        alpha=0.9,
        edgecolor='white',  # a hairline outline, or a deep (dark) quake vanishes into the ocean
        linewidth=0.3,
        altitude=0.02,  # lift them clear of the sphere's own facets
    )
    # matplotlib's 3-D axes reserves room for a cube's diagonal, so a sphere sits small
    # in a lot of white. Give the axes the figure (less a strip for the colour bar) and
    # zoom modestly: measured, this takes the disc from 55% to 61% of the frame width
    # while staying round. zoom past ~1.7 crops the sphere into a squircle -- and the
    # axes limits clip it before it reaches the frame border, so an edge-contact check
    # does not catch that.
    globe.ax.set_position([0.0, 0.0, 0.80, 1.0])
    globe.ax.set_box_aspect((1, 1, 1), zoom=1.35)
    bar = globe.fig.colorbar(quakes, ax=globe.ax, shrink=0.6, pad=0.02)
    bar.set_label('hypocentre depth (km)')
    globe.ax.set_title('Global M5+ seismicity, 2023')
    globe.save(str(OUT_DIR / 'seismicity_globe.png'), dpi=110)
    globe.close()
    display(Image(filename=str(OUT_DIR / 'seismicity_globe.png')))

### Spinning it

One revolution carries the whole circum-Pacific belt past the viewer — the Andes and the Aleutians as well
as the arcs facing us above — which no single view of a sphere can show.

`TexturedGlobe.animate` sweeps the rotation itself, and it turns the overlay with the sphere: the epicentres
`points` drew are re-placed at each frame's spin, so every quake stays on the ground it happened on. (Pass
`spin=` to `points` instead to pin an overlay in place.) Earlier releases turned only the sphere, which is
why this used to be a hand-rolled frame loop redrawing both. The framing is applied once, before the call,
because `animate` keeps the axes it is given. 36 frames at 6 fps is one turn in six seconds.

In [ ]:
if events is not None and len(events):
    N_FRAMES, FPS = 36, 6

    globe = de.TexturedGlobe.from_provider(
        # n_lon/n_lat are the SPHERE MESH, and the mesh -- not the texture -- is what
        # limits detail: measured on this figure, 140x70 scores 1.15 mean edge energy
        # against 1.50 at 360x180. (GIF quantisation costs nothing here: the same frame
        # scores 1.50 as a PNG and 1.50 palette-reduced.)
        'Esri.WorldImagery',
        texture_n_lon=1440,
        texture_n_lat=720,
        n_lon=360,
        n_lat=180,
        figsize=(6, 6),
    )
    globe.draw(spin=231.0)
    # Deliberately no `spin=`: left unpinned, the overlay is turned to each frame's spin.
    globe.points(
        events,
        s=2**magnitude / 2,
        c=depth,
        cmap='viridis_r',
        alpha=0.9,
        edgecolor='white',
        linewidth=0.3,
        altitude=0.02,
    )
    # Applied once rather than per frame -- `animate` keeps the axes position and aspect.
    globe.ax.set_position([0.0, 0.0, 1.0, 1.0])
    globe.ax.set_box_aspect((1, 1, 1), zoom=1.5)

    anim = globe.animate(
        n_frames=N_FRAMES, revolutions=1, start_spin=231.0, interval=1000 / FPS
    )
    # One render, two files: the GIF is derived from the written MP4, not redrawn.
    spin_mp4, spin_gif = save_animation(
        anim,
        str(OUT_DIR / 'seismicity_globe.mp4'),
        fps=FPS,
        gif=str(OUT_DIR / 'seismicity_globe.gif'),
    )
    plt.close('all')
    globe.close()
    display(Image(filename=spin_gif))